# Lab 11.1 (Databricks edition) — MovieLens data processing with PySpark

**Modern Data Architectures for Big Data I · IE University**

The same lab as *MovieLens MapReduce data processing*, but on **Databricks** with **PySpark**:

| Question | Hadoop version | This notebook |
|---|---|---|
| Number of movies by genre | `genres-mapper.py` + `genres-reducer.py` + Hadoop Streaming | `explode` + `groupBy` + `count` |
| Average rating by movie | `movie-rating-mapper.py` + `movie-rating-reducer.py` | `groupBy` + `avg` + `count` |

**Same data storage layout as the slides.** In OSBDET the data lake lives in HDFS under `/datalake/raw/movielens/<dataset>/` and job results under `/jobs/<job>/`.
In Databricks we mirror it with **Unity Catalog volumes**:

| OSBDET (HDFS) | Databricks (Unity Catalog volume) |
|---|---|
| `/datalake/raw/movielens/movies/movies.csv` | `/Volumes/workspace/datalake/raw/movielens/movies/movies.csv` |
| `/datalake/raw/movielens/ratings/ratings.csv` | `/Volumes/workspace/datalake/raw/movielens/ratings/ratings.csv` |
| `/jobs/movies-genre` | `/Volumes/workspace/datalake/jobs/movies-genre` |
| `/jobs/movie-rating` | `/Volumes/workspace/datalake/jobs/movie-rating` |

> Works on **Databricks Free Edition** (serverless compute). Run the cells **in order**.

## 0. Configuration

`workspace` is the default catalog in Free Edition. Change `CATALOG` if your workspace uses another one (e.g. `main`).

In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"     # Unity Catalog catalog
SCHEMA  = "datalake"      # plays the role of the /datalake folder

RAW  = f"/Volumes/{CATALOG}/{SCHEMA}/raw/movielens"   # = /datalake/raw/movielens in HDFS
JOBS = f"/Volumes/{CATALOG}/{SCHEMA}/jobs"            # = /jobs in HDFS

print("raw zone :", RAW)
print("job output:", JOBS)

## 1. Prepare the storage (run once)

Creates the schema `datalake`, two volumes (`raw` and `jobs`) and the dataset folders, exactly like the HDFS layout from lab 8.1.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.jobs")

for dataset in ["movies", "ratings", "tags", "links"]:
    dbutils.fs.mkdirs(f"{RAW}/{dataset}")

display(dbutils.fs.ls(RAW))

## 2. Put the MovieLens files in the raw zone

**Option A — upload (recommended, always works).** Free Edition only allows internet access to a limited set of trusted domains, so uploading is the safest route.

1. Open **Catalog** (left sidebar) → `workspace` → `datalake` → `raw`
2. Click **Upload to this volume**
3. Upload `movies.csv` into `movielens/movies/` and `ratings.csv` into `movielens/ratings/` (they are in `movielens-resources.zip` from lab 8.1)

**Option B — download from GroupLens.** Run the next cell. If your workspace blocks the download, it tells you, and you use option A.

In [ ]:
import os, shutil, urllib.request, zipfile

URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
landing = f"/Volumes/{CATALOG}/{SCHEMA}/raw/_landing"
os.makedirs(landing, exist_ok=True)
zip_path = f"{landing}/ml-latest-small.zip"

try:
    urllib.request.urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        for dataset in ["movies", "ratings", "tags", "links"]:
            with z.open(f"ml-latest-small/{dataset}.csv") as src, \
                 open(f"{RAW}/{dataset}/{dataset}.csv", "wb") as dst:
                shutil.copyfileobj(src, dst)
    print("Downloaded and copied into the raw zone ✓")
except Exception as e:
    print("Download not possible here:", type(e).__name__, "→ use option A (upload).")

## 3. Data set availability verification

Same check as in the Hadoop lab (`hdfs dfs -ls`): both files must be there.

In [ ]:
for dataset in ["movies", "ratings"]:
    files = [f for f in dbutils.fs.ls(f"{RAW}/{dataset}") if f.name.endswith(".csv")]
    assert files, f"No CSV in {RAW}/{dataset}: upload it first (step 2)"
    for f in files:
        print(f"✓ {f.path}  {f.size:,} bytes")

## 4. Load the data as DataFrames

Spark reads **every file in the folder**, just like `-input /datalake/raw/movielens/movies` in Hadoop.
`header=True` skips the header line, which the Python mapper had to do by hand. Spark's CSV reader also handles titles with commas inside quotes.

In [ ]:
movies = spark.read.csv(f"{RAW}/movies", header=True, inferSchema=True)
ratings = spark.read.csv(f"{RAW}/ratings", header=True, inferSchema=True)

movies.printSchema()
print(f"{movies.count():,} movies · {ratings.count():,} ratings")
display(movies.limit(5))

## 5. Number of movies by genre

### MapReduce thinking, in DataFrames

| MapReduce | PySpark |
|---|---|
| **map**: one `(genre, 1)` per genre of each movie | `F.explode(F.split("genres", "\\|"))`: one row per genre |
| **sort / shuffle**: group pairs by genre | `groupBy("genre")` |
| **reduce**: count the 1s | `.count()` |

In [ ]:
# "map": one row per (movie, genre)
movie_genre = movies.select("movieId", F.explode(F.split("genres", r"\|")).alias("genre"))
display(movie_genre.limit(5))      # Toy Story → Adventure, Animation, Children, Comedy, Fantasy

In [ ]:
# "sort + reduce": group by genre and count
genre_counts = (movie_genre
                .groupBy("genre")
                .agg(F.count("*").alias("movies"))
                .orderBy(F.desc("movies")))

display(genre_counts)              # tip: click "+" → Visualization → Bar chart

### The same with SQL

Exactly the `GROUP BY` from the lecture.

In [ ]:
movies.createOrReplaceTempView("movies")

display(spark.sql("""
    SELECT genre, COUNT(*) AS movies
    FROM (SELECT explode(split(genres, '\\\\|')) AS genre FROM movies)
    GROUP BY genre
    ORDER BY movies DESC
"""))

### Under the hood: where is the "sort"?

`explain()` shows the physical plan. Look for **`Exchange hashpartitioning(genre…)`**: that's the **shuffle**, Spark's version of MapReduce's sort phase. Everything before it is the "map" side; the final aggregate is the "reduce" side.

In [ ]:
genre_counts.explain()

### Save the results (like `/jobs/movies-genre`)

Two ways: a **CSV folder** in the `jobs` volume (same as Hadoop's `part-00000`) and a **table** in Unity Catalog that anyone can query.

In [ ]:
(genre_counts.coalesce(1)                 # one output file, like one reducer
    .write.mode("overwrite")
    .option("header", True)
    .csv(f"{JOBS}/movies-genre"))

genre_counts.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.movies_genre")

display(dbutils.fs.ls(f"{JOBS}/movies-genre"))   # _SUCCESS + part-00000-….csv

## 6. Average rating by movie

| MapReduce | PySpark |
|---|---|
| **map**: `(movieId, rating)` | `select("movieId", "rating")` (implicit) |
| **sort / shuffle**: group by movie | `groupBy("movieId")` |
| **reduce**: running sum and count → average | `agg(avg, count)` |

> The original lab called this "average rating *by user*", but the key is the **movie**: it's the average rating **of each movie**.

In [ ]:
movie_rating = (ratings
                .groupBy("movieId")
                .agg(F.round(F.avg("rating"), 2).alias("avg_rating"),
                     F.count("*").alias("num_ratings"))
                .orderBy("movieId"))

print(f"{movie_rating.count():,} movies have at least one rating")   # 9,724
display(movie_rating.limit(5))     # movie 1 (Toy Story): 3.92 from 215 ratings

In [ ]:
(movie_rating.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(f"{JOBS}/movie-rating"))

movie_rating.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.movie_rating")

### Business insight: averages need enough ratings

A movie with a single 5-star rating isn't "the best movie". Join with `movies` for titles and keep movies with at least 100 ratings.

In [ ]:
top_movies = (movie_rating
              .filter("num_ratings >= 100")
              .join(movies, "movieId")
              .orderBy(F.desc("avg_rating"))
              .select("title", "avg_rating", "num_ratings"))

display(top_movies.limit(10))      # Shawshank Redemption 4.43 (317) · Godfather 4.29 (192) · Fight Club 4.27 (218)

## 7. Challenges

**A · Average rating by user** (the original lab title). Who is the toughest critic?

In [ ]:
user_rating = (ratings.groupBy("userId")
               .agg(F.round(F.avg("rating"), 2).alias("avg_rating"),
                    F.count("*").alias("num_ratings"))
               .filter("num_ratings >= 20")
               .orderBy("avg_rating"))

display(user_rating.limit(5))

**B · Ratings per year.** `timestamp` is in seconds since 1970: convert it with `F.from_unixtime` and `F.year`, then count per year.

**C · Best genre.** Join `movie_genre` with `movie_rating` and compute the average rating per genre (weighted by `num_ratings`).

In [ ]:
# B · your code here

In [ ]:
# C · your code here

## 8. Clean up (optional)

Remove the job outputs and tables. Keep the raw zone: raw data is never deleted.

In [ ]:
# dbutils.fs.rm(f"{JOBS}/movies-genre", recurse=True)
# dbutils.fs.rm(f"{JOBS}/movie-rating", recurse=True)
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.movies_genre")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.movie_rating")